# 02 — Primer modelo LightGBM

Entrena el primer modelo BiciMAD, lo compara con el baseline naive y analiza las features más importantes.

> **Nota:** Necesitas al menos 30 días de datos para splits train/val/test completos (28+1+1). Con menos datos, usa  en el CLI.

In [ ]:
import sys
from pathlib import Path

# Find repo root regardless of where Jupyter was launched from
_cwd = Path.cwd()
REPO_ROOT = _cwd
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import polars as pl
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.figsize"] = (12, 5)
plt.style.use("seaborn-v0_8-whitegrid")

DATA_DIR = str(REPO_ROOT / "data/raw")
MODEL_DIR = str(REPO_ROOT / "data/models")
print(f"Repo root: {REPO_ROOT}")

## 1. Cargar dataset

In [ ]:
from src.features.build_dataset import build_training_dataset

df = build_training_dataset(source="local", data_dir=DATA_DIR)
print(f"Filas totales: {len(df):,}  |  Columnas: {len(df.columns)}")
print(f"Estaciones: {df['station_id'].n_unique()}")
print(f"Rango temporal: {df['snapshot_timestamp'].min()} → {df['snapshot_timestamp'].max()}")

## 2. Split temporal (28 días train / 1 val / 1 test)

In [ ]:
from src.training.split import temporal_split

train_df, val_df, test_df = temporal_split(df, train_days=28, val_days=1, test_days=1)
print(f"Train: {len(train_df):,} filas  [{train_df['snapshot_timestamp'].min()} → {train_df['snapshot_timestamp'].max()}]")
print(f"Val:   {len(val_df):,} filas  [{val_df['snapshot_timestamp'].min()} → {val_df['snapshot_timestamp'].max()}]")
print(f"Test:  {len(test_df):,} filas  [{test_df['snapshot_timestamp'].min()} → {test_df['snapshot_timestamp'].max()}]")

## 3. Baseline naive

In [ ]:
from src.training.baseline import naive_baseline

baseline = naive_baseline(df.filter(pl.col("target_dock_bikes_1h").is_not_null()))
print(f"Baseline MAE:  {baseline['mae']:.4f}")
print(f"Baseline RMSE: {baseline['rmse']:.4f}")

## 4. Entrenar LightGBM (parámetros por defecto)

In [ ]:
from src.training.train import train_model

model = train_model(train_df, val_df)
print(f"Árboles entrenados: {model.num_trees()}  |  Mejor iteración: {model.best_iteration}")

## 5. Evaluar en test

In [ ]:
from src.training.evaluate import evaluate, evaluate_critical_states

metrics = evaluate(model, test_df)
print(f"Test MAE:          {metrics['mae']:.4f}")
print(f"Test RMSE:         {metrics['rmse']:.4f}")
print(f"R²:                {metrics['r2']:.4f}")
print(f"Baseline MAE:      {metrics['baseline_mae']:.4f}")
print(f"Mejora vs baseline: {metrics['improvement_pct']:.1f}%")

critical = evaluate_critical_states(model, test_df)
print(f"
Estaciones vacías — Precisión: {critical['empty_precision']:.3f}  Recall: {critical['empty_recall']:.3f}")
print(f"Estaciones llenas — Precisión: {critical['full_precision']:.3f}  Recall: {critical['full_recall']:.3f}")

## 6. Feature importance (Top 20)

In [ ]:
import pandas as pd

importance = pd.DataFrame({
    "feature": model.feature_name(),
    "importance": model.feature_importance(importance_type="gain"),
}).sort_values("importance", ascending=False).head(20)

plt.barh(importance["feature"][::-1], importance["importance"][::-1], color="steelblue")
plt.xlabel("Importance (gain)")
plt.title("Top 20 features por importancia")
plt.tight_layout()
plt.show()

## 7. Distribución de errores por hora

In [ ]:
from src.training.train import _prepare_features

X_test, y_test = _prepare_features(test_df.filter(pl.col("target_dock_bikes_1h").is_not_null()))
preds = model.predict(X_test)
errors = abs(y_test.to_numpy() - preds)

# Add hour_of_day back for grouping
test_valid = test_df.filter(pl.col("target_dock_bikes_1h").is_not_null())
hourly_mae = (
    test_valid.with_columns(pl.lit(errors.tolist()).alias("abs_error"))
    .group_by("hour_of_day")
    .agg(pl.col("abs_error").mean().alias("mae"))
    .sort("hour_of_day")
)
plt.bar(hourly_mae["hour_of_day"].to_numpy(), hourly_mae["mae"].to_numpy(), color="steelblue")
plt.xlabel("Hora del día")
plt.ylabel("MAE")
plt.title("MAE por hora del día (test set)")
plt.xticks(range(0, 24))
plt.show()

## 8. Guardar modelo

In [ ]:
from pathlib import Path
from src.training.registry import save_model
from src.training.evaluate import generate_report

version_dir = save_model(model, metrics, output_dir=Path(MODEL_DIR))
print(f"Modelo guardado en: {version_dir}")

generate_report(metrics, version_dir / "evaluation_report.json", model=model)
print(f"Reporte guardado en: {version_dir / 'evaluation_report.json'}")